# Avro Basics — 09: Avro to Parquet Pipeline


The production pattern: Avro arrives as raw/landing data,
Spark transforms and writes it as Parquet for analytics.

Topics: multi-version Avro ingestion, schema normalization, type conversion,
partitioned Parquet output, validation, incremental processing.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:07:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Setup: Multi-Version Avro Landing Zone



In [2]:

import json as pyjson, pathlib, random
random.seed(42)

LANDING = f"{DATA_DIR}/landing"
pathlib.Path(LANDING).mkdir(exist_ok=True)

S_V1 = pyjson.dumps({"type":"record","name":"Order","fields":[
    {"name":"order_id","type":"long"},
    {"name":"customer_id","type":"long"},
    {"name":"product","type":"string"},
    {"name":"amount","type":"double"},
    {"name":"status","type":"string"},
    {"name":"created_at","type":"string"},  # date as string in v1
]})

S_V2 = pyjson.dumps({"type":"record","name":"Order","fields":[
    {"name":"order_id","type":"long"},
    {"name":"customer_id","type":"long"},
    {"name":"product","type":"string"},
    {"name":"amount","type":"double"},
    {"name":"status","type":"string"},
    {"name":"created_at","type":"string"},
    {"name":"discount",  "type":["null","double"],"default":None},
    {"name":"region",    "type":"string","default":"UNKNOWN"},
]})

import datetime as dt
def make_orders(n, start_id, schema_v):
    base = dt.date(2024,1,1)
    rows = []
    for i in range(n):
        d = base + dt.timedelta(days=random.randint(0,180))
        row = [start_id+i, random.randint(1,1000),
               f"Product_{random.randint(1,50)}",
               round(random.uniform(10,500),2),
               random.choice(["pending","confirmed","shipped"]),
               str(d)]
        if schema_v == 2:
            row += [round(random.uniform(0,0.3),2) if random.random()>0.5 else None,
                    random.choice(["AMER","EMEA","APAC"])]
        rows.append(tuple(row))
    return rows

# Write 3 months of V1 + 2 months of V2
for month in [1,2,3]:
    spark.createDataFrame(make_orders(2000,month*2000,1), ["order_id","customer_id","product","amount","status","created_at"]) \
         .write.format("avro").option("avroSchema",S_V1) \
         .mode("overwrite").save(f"{LANDING}/month={month:02d}_v1")

for month in [4,5]:
    spark.createDataFrame(make_orders(2000,month*2000,2), ["order_id","customer_id","product","amount","status","created_at","discount","region"]) \
         .write.format("avro").option("avroSchema",S_V2) \
         .mode("overwrite").save(f"{LANDING}/month={month:02d}_v2")

print("Landing zone created:")
import os
for d in sorted(os.listdir(LANDING)):
    cnt = spark.read.format("avro").load(f"{LANDING}/{d}").count()
    print(f"  {d}/  {cnt:,} records")


Landing zone created:
  month=01_v1/  2,000 records
  month=02_v1/  2,000 records
  month=03_v1/  2,000 records
  month=04_v2/  2,000 records
  month=05_v2/  2,000 records


## Step 2 — Read All Versions with Wide Schema



In [3]:

# Wide schema covers both V1 and V2
WIDE_SCHEMA = pyjson.dumps({"type":"record","name":"Order","fields":[
    {"name":"order_id",   "type":"long"},
    {"name":"customer_id","type":"long"},
    {"name":"product",    "type":"string"},
    {"name":"amount",     "type":"double"},
    {"name":"status",     "type":"string"},
    {"name":"created_at", "type":"string"},
    {"name":"discount",   "type":["null","double"],"default":None},
    {"name":"region",     "type":"string","default":"UNKNOWN"},
]})

# Read all months — Avro handles missing fields via defaults
all_dirs = [f"{LANDING}/{d}" for d in sorted(os.listdir(LANDING))]
df_raw = spark.read.format("avro").option("avroSchema",WIDE_SCHEMA).load(all_dirs)
print(f"Total raw records: {df_raw.count():,}")
df_raw.groupBy("region").count().orderBy("region").show()


Total raw records: 10,000
+-------+-----+
| region|count|
+-------+-----+
|   AMER| 1298|
|   APAC| 1325|
|   EMEA| 1377|
|UNKNOWN| 6000|
+-------+-----+



## Step 3 — Transform and Normalize



In [4]:

# Type conversions and derived columns
from pyspark.sql.functions import to_date

df_clean = df_raw \
    .withColumn("order_date",  to_date(F.col("created_at"), "yyyy-MM-dd")) \
    .withColumn("discount",    F.coalesce(F.col("discount"), F.lit(0.0))) \
    .withColumn("net_amount",  F.round(F.col("amount") * (1 - F.col("discount")), 2)) \
    .withColumn("year",        F.year("order_date")) \
    .withColumn("month",       F.month("order_date")) \
    .drop("created_at")

print("After transformation:")
df_clean.printSchema()
df_clean.show(5)


After transformation:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- discount: double (nullable = false)
 |-- region: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+--------+-----------+----------+------+---------+--------+------+----------+----------+----+-----+
|order_id|customer_id|   product|amount|   status|discount|region|order_date|net_amount|year|month|
+--------+-----------+----------+------+---------+--------+------+----------+----------+----+-----+
|   11500|        128| Product_5| 63.24|confirmed|     0.0|  AMER|2024-04-15|     63.24|2024|    4|
|   11501|        456|Product_14|135.12|  shipped|     0.0|  AMER|2024-01-07|    135.12|2024|    1|
|   11502|        320| Product_8|147.48|co

## Step 4 — Write Partitioned Parquet + Validate



In [5]:

STORAGE = f"{DATA_DIR}/storage_parquet"
df_clean.write.mode("overwrite") \
              .partitionBy("region","year","month") \
              .option("compression","zstd") \
              .parquet(STORAGE)

# Validate row counts
raw_count     = df_raw.count()
parquet_count = spark.read.parquet(STORAGE).count()
print(f"Validation:")
print(f"  Raw Avro  : {raw_count:,} rows")
print(f"  Parquet   : {parquet_count:,} rows")
print(f"  Match     : {'✅ PASS' if raw_count==parquet_count else '❌ FAIL'}")
print()

# Revenue by region
spark.read.parquet(STORAGE) \
     .groupBy("region").agg(F.sum("net_amount").alias("revenue"), F.count("*").alias("orders")) \
     .orderBy(F.desc("revenue")).show()


Validation:
  Raw Avro  : 10,000 rows
  Parquet   : 10,000 rows
  Match     : ✅ PASS



[Stage 36:>                                                         (0 + 4) / 4]

+-------+------------------+------+
| region|           revenue|orders|
+-------+------------------+------+
|UNKNOWN| 1537746.920000003|  6000|
|   EMEA|320031.44000000006|  1377|
|   APAC|         305486.85|  1325|
|   AMER|         303288.77|  1298|
+-------+------------------+------+



## Summary



In [6]:

print("""
Avro → Parquet pipeline pattern:

  1. Landing zone (raw Avro)
     - Keep original Avro files unchanged
     - Read with wide schema covering all versions

  2. Transformation
     - Type conversions (string dates → DateType)
     - Fill nulls with sensible defaults
     - Derive computed columns

  3. Storage zone (Parquet)
     - Partitioned by common filter columns
     - Compressed (zstd recommended)
     - Validated (row count check)

  4. Analytics
     - Query Parquet (fast, columnar)
     - Keep Avro as archive (schema history)

Pipeline schedule:
  Incremental: track last processed file/batch
  Full reload: reprocess landing zone when schema changes
""")



Avro → Parquet pipeline pattern:

  1. Landing zone (raw Avro)
     - Keep original Avro files unchanged
     - Read with wide schema covering all versions

  2. Transformation
     - Type conversions (string dates → DateType)
     - Fill nulls with sensible defaults
     - Derive computed columns

  3. Storage zone (Parquet)
     - Partitioned by common filter columns
     - Compressed (zstd recommended)
     - Validated (row count check)

  4. Analytics
     - Query Parquet (fast, columnar)
     - Keep Avro as archive (schema history)

Pipeline schedule:
  Incremental: track last processed file/batch
  Full reload: reprocess landing zone when schema changes

